# Logistic Regression

Linear model for binary and multiclass classification via log-odds. The workhorse of classification — highly interpretable with well-understood statistical properties. This template covers L1, L2, and ElasticNet regularization variants.

**When to Use:**
- Binary or multiclass classification problems
- Interpretability is important (odds ratios, coefficient analysis)
- Baseline classification model before trying complex methods
- When probability estimates are needed (well-calibrated outputs)
- Linearly separable or near-linearly separable features

**Key Assumptions / Considerations:**
- Linear relationship between features and log-odds of the outcome
- Observations are independent
- No severe multicollinearity among predictors
- Large sample sizes improve reliability of coefficient estimates
- Regularization (L1/L2/ElasticNet) helps with multicollinearity and feature selection

**References:**
- [sklearn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [sklearn ROC Curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)
- [sklearn Precision-Recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html)

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, log_loss, classification_report, 
    confusion_matrix, roc_curve, precision_recall_curve
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data 

np.random.seed(100)
n_samples = 8000

# numeric
age = np.random.randint(18, 70, n_samples)
income = np.random.normal(50000, 15000, n_samples)
num_transactions = np.random.poisson(10, n_samples)

# categorical
gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

# continuous target (used to derive binary)
target_continuous = 0.05 * age + 0.0005 * income + 0.3 * num_transactions + np.random.normal(0, 2, n_samples)

# binary classification target
target_binary = (target_continuous > np.median(target_continuous)).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
    'target': target_binary
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Target Distribution

plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title("Target Variable Distribution")
plt.show()

print("Class balance:")
print(y.value_counts(normalize=True))

In [ ]:
# Train/Test Split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Preprocessing

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Parameter Grids

models = {
    "LogReg_L2": (LogisticRegression(solver="lbfgs", penalty="l2", max_iter=2000, random_state=42), {
        "clf__C": np.logspace(-4, 2, 10),
    }),
    "LogReg_L1": (LogisticRegression(solver="saga", penalty="l1", max_iter=2000, random_state=42), {
        "clf__C": np.logspace(-4, 2, 10),
    }),
    "LogReg_ElasticNet": (LogisticRegression(solver="saga", penalty="elasticnet", max_iter=2000, random_state=42), {
        "clf__C": np.logspace(-4, 2, 10),
        "clf__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
    }),
}

# Cross Validation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Training

def train_models(X_train, y_train, X_test, y_test):

    results = []
    fitted_pipelines = []

    for name, (model, params) in models.items():
        print(f"\n\U0001f539 Training {name}...")

        pipe = Pipeline([("prep", preprocessor), ("clf", model)])

        grid = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring="roc_auc", return_train_score=False)

        try:
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_
            fitted_pipelines.append((name, best_model))
            
            y_pred = best_model.predict(X_test)
            y_proba = best_model.predict_proba(X_test)[:, 1]

            metrics = {
                "Model": name,
                "Best Params": grid.best_params_,
                "Accuracy": accuracy_score(y_test, y_pred),
                "Recall": recall_score(y_test, y_pred),
                "Precision": precision_score(y_test, y_pred),
                "F1 Score": f1_score(y_test, y_pred),
                "ROC-AUC": roc_auc_score(y_test, y_proba),
                "Log Loss": log_loss(y_test, y_proba),
            }

            results.append(metrics)
        except Exception as e:
            print(f"\u26a0\ufe0f Skipping {name} due to error: {e}")
            continue

    return results, fitted_pipelines

In [ ]:
results, pipelines = train_models(X_train_full, y_train_full, X_test, y_test)

In [ ]:
# Results Summary

results_df = pd.DataFrame(results)
print("\n\u2705 Model Evaluation Summary:")
results_df

In [ ]:
# Best Model by ROC-AUC
best_model_name = max(results, key=lambda x: x['ROC-AUC'])['Model']
best_model_pipeline = [p for n, p in pipelines if n == best_model_name][0]
print(f"\n\U0001f3c6 Best Model: {best_model_name}")

In [ ]:
# Coefficients & Odds Ratios

for name, pipe in pipelines:
    print(f"\nModel: {name}")
    
    clf = pipe.named_steps["clf"]
    coef = clf.coef_.flatten()
    intercept = clf.intercept_[0]
    
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    df_coef = pd.DataFrame({
        "feature": feature_names,
        "coef": coef,
        "odds_ratio": np.exp(coef)
    }).sort_values("coef", key=abs, ascending=False)
    
    # Add intercept row
    df_intercept = pd.DataFrame({
        "feature": ["Intercept"],
        "coef": [intercept],
        "odds_ratio": [np.exp(intercept)]
    })
    df_coef = pd.concat([df_intercept, df_coef], ignore_index=True)
    
    print(df_coef)
    
    # Bar chart (exclude intercept)
    plot_df = df_coef[df_coef["feature"] != "Intercept"]
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="coef", y="feature")
    plt.title(f"{name} - Coefficients")
    plt.tight_layout()
    plt.show()

In [ ]:
# ROC Curves (all models overlaid)

plt.figure(figsize=(8, 6))

for name, pipe in pipelines:
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")

plt.plot([0, 1], [0, 1], 'r--', label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curves

plt.figure(figsize=(8, 6))

for name, pipe in pipelines:
    y_proba = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    plt.plot(rec, prec, label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Threshold Optimization

y_proba = best_model_pipeline.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t))

best_threshold = thresholds[np.argmax(f1_scores)]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, f1_scores)
plt.axvline(best_threshold, color='r', linestyle='--', label=f"Best threshold: {best_threshold:.2f}")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Classification Threshold")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\U0001f3c6 Optimal threshold: {best_threshold:.2f}")
print(f"   F1 at optimal threshold: {max(f1_scores):.4f}")

In [ ]:
# Classification Diagnostics

def classification_diagnostics(model, X, y_true, name="Model"):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title(f"{name} - Confusion Matrix")
    
    # Predicted Probability Distribution
    axes[1].hist(y_proba[y_true == 0], bins=30, alpha=0.5, label="Class 0")
    axes[1].hist(y_proba[y_true == 1], bins=30, alpha=0.5, label="Class 1")
    axes[1].set_xlabel("Predicted Probability")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"{name} - Probability Distribution")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(classification_report(y_true, y_pred))

classification_diagnostics(best_model_pipeline, X_test, y_test, best_model_name)

In [ ]:
# Regularization Path

C_values = np.logspace(-4, 2, 30)
coefs = []

for C in C_values:
    lr = LogisticRegression(solver="saga", penalty="l1", C=C, max_iter=2000, random_state=42)
    pipe = Pipeline([("prep", preprocessor), ("clf", lr)])
    pipe.fit(X_train_full, y_train_full)
    coefs.append(pipe.named_steps["clf"].coef_.flatten())

coefs = np.array(coefs)
feature_names = preprocessor.fit(X_train_full).get_feature_names_out()

plt.figure(figsize=(10, 6))
for i, fname in enumerate(feature_names):
    plt.plot(np.log10(C_values), coefs[:, i], label=fname)

plt.xlabel("log10(C)")
plt.ylabel("Coefficient Value")
plt.title("L1 Regularization Path")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# future work:
#   multiclass: set multi_class='multinomial' for >2 classes
#   class imbalance: use class_weight='balanced' or SMOTE
#   calibration: sklearn.calibration.CalibratedClassifierCV for better probability estimates
#   polynomial features: add interaction terms with PolynomialFeatures
#   compare with SGDClassifier(loss='log_loss') for large-scale datasets